In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
import sklearn

from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

import matplotlib.pyplot as plt

print("TensorFlow:", tf.__version__)
print("Scikit-learn:", sklearn.__version__)
print("Libraries imported successfully!")

TensorFlow: 2.21.0
Scikit-learn: 1.6.1
Libraries imported successfully!


In [2]:
url = "https://raw.githubusercontent.com/npradaschnor/Pima-Indians-Diabetes-Dataset/master/diabetes.csv"

df = pd.read_csv(url)

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)

display(df.head())

Dataset loaded successfully!
Dataset shape: (768, 9)


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [3]:
print("Column names:")
print(df.columns.tolist())

print("\nDataset information:")
df.info()

print("\nMissing values:")
print(df.isnull().sum())

print("\nOutcome distribution:")
print(df["Outcome"].value_counts())


Column names:
['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome']

Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-null    int64  
 1   Glucose                   768 non-null    int64  
 2   BloodPressure             768 non-null    int64  
 3   SkinThickness             768 non-null    int64  
 4   Insulin                   768 non-null    int64  
 5   BMI                       768 non-null    float64
 6   DiabetesPedigreeFunction  768 non-null    float64
 7   Age                       768 non-null    int64  
 8   Outcome                   768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB

Missing values:
Pregnancies                 0
Glucose                     0
Bl

In [4]:
X = df.drop("Outcome", axis=1).copy()
y = df["Outcome"].copy()

print("Input features:", X.shape)
print("Target:", y.shape)

Input features: (768, 8)
Target: (768,)


In [5]:
zero_columns = [
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI"
]

X[zero_columns] = X[zero_columns].replace(0, np.nan)

print("Invalid zero values handled.")
print("\nMissing values in each column:")
print(X.isnull().sum())

Invalid zero values handled.

Missing values in each column:
Pregnancies                   0
Glucose                       5
BloodPressure                35
SkinThickness               227
Insulin                     374
BMI                          11
DiabetesPedigreeFunction      0
Age                           0
dtype: int64


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

Training samples: 614
Testing samples: 154


In [7]:
class KerasBinaryClassifier(ClassifierMixin, BaseEstimator):

    def __init__(
        self,
        neurons=16,
        activation="relu",
        kernel_initializer="he_normal",
        dropout_rate=0.0,
        learning_rate=0.001,
        batch_size=32,
        epochs=20,
        verbose=0,
        random_state=42
    ):
        self.neurons = neurons
        self.activation = activation
        self.kernel_initializer = kernel_initializer
        self.dropout_rate = dropout_rate
        self.learning_rate = learning_rate
        self.batch_size = batch_size
        self.epochs = epochs
        self.verbose = verbose
        self.random_state = random_state

    def fit(self, X, y):
        X = np.asarray(X, dtype=np.float32)
        y = np.asarray(y, dtype=np.int32)

        tf.keras.backend.clear_session()
        tf.keras.utils.set_random_seed(self.random_state)

        self.n_features_in_ = X.shape[1]
        self.classes_ = np.unique(y)

        self.model_ = tf.keras.Sequential([
            tf.keras.layers.Input(shape=(self.n_features_in_,)),

            tf.keras.layers.Dense(
                self.neurons,
                activation=self.activation,
                kernel_initializer=self.kernel_initializer
            ),

            tf.keras.layers.Dropout(self.dropout_rate),

            tf.keras.layers.Dense(
                1,
                activation="sigmoid"
            )
        ])

        self.model_.compile(
            optimizer=tf.keras.optimizers.Adam(
                learning_rate=self.learning_rate
            ),
            loss="binary_crossentropy",
            metrics=["accuracy"]
        )

        self.model_.fit(
            X,
            y,
            batch_size=self.batch_size,
            epochs=self.epochs,
            verbose=self.verbose
        )

        return self

    def predict_proba(self, X):
        X = np.asarray(X, dtype=np.float32)
        probabilities = self.model_.predict(X, verbose=0).ravel()

        return np.column_stack([
            1 - probabilities,
            probabilities
        ])

    def predict(self, X):
        probabilities = self.predict_proba(X)[:, 1]
        return (probabilities >= 0.5).astype(int)

In [8]:
pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("classifier", KerasBinaryClassifier())
])

print("Preprocessing and deep learning pipeline created.")

Preprocessing and deep learning pipeline created.


In [9]:
test_grid = GridSearchCV(
    estimator=pipeline,
    param_grid={
        "classifier__neurons": [16, 32]
    },
    scoring="accuracy",
    cv=2,
    n_jobs=1,
    verbose=2,
    error_score="raise"
)

test_grid.fit(X_train, y_train)

print("\nTest grid search successful!")
print("Best parameters:", test_grid.best_params_)
print("Best CV accuracy:", round(test_grid.best_score_ * 100, 2), "%")

Fitting 2 folds for each of 2 candidates, totalling 4 fits
[CV] END .............................classifier__neurons=16; total time=   6.1s
[CV] END .............................classifier__neurons=16; total time=   2.3s
[CV] END .............................classifier__neurons=32; total time=   2.3s
[CV] END .............................classifier__neurons=32; total time=   3.3s

Test grid search successful!
Best parameters: {'classifier__neurons': 16}
Best CV accuracy: 74.59 %


In [10]:
param_grid = {
    "classifier__batch_size": [16, 32],
    "classifier__epochs": [20, 30],
    "classifier__learning_rate": [0.001, 0.01],
    "classifier__kernel_initializer": ["uniform", "he_normal"],
    "classifier__activation": ["relu", "tanh"],
    "classifier__dropout_rate": [0.0, 0.2],
    "classifier__neurons": [16, 32]
}

print("Hyperparameter combinations:", 128)

Hyperparameter combinations: 128


In [ ]:
grid = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring="accuracy",
    cv=3,
    n_jobs=1,
    verbose=2,
    refit=True,
    error_score="raise"
)

grid.fit(X_train, y_train)

print("\nGrid search completed successfully!")

Fitting 3 folds for each of 128 candidates, totalling 384 fits
[CV] END classifier__activation=relu, classifier__batch_size=16, classifier__dropout_rate=0.0, classifier__epochs=20, classifier__kernel_initializer=uniform, classifier__learning_rate=0.001, classifier__neurons=16; total time=   2.8s


[CV] END classifier__activation=relu, classifier__batch_size=16, classifier__dropout_rate=0.0, classifier__epochs=20, classifier__kernel_initializer=uniform, classifier__learning_rate=0.001, classifier__neurons=16; total time=   4.9s


[CV] END classifier__activation=relu, classifier__batch_size=16, classifier__dropout_rate=0.0, classifier__epochs=20, classifier__kernel_initializer=uniform, classifier__learning_rate=0.001, classifier__neurons=16; total time=   3.6s
[CV] END classifier__activation=relu, classifier__batch_size=16, classifier__dropout_rate=0.0, classifier__epochs=20, classifier__kernel_initializer=uniform, classifier__learning_rate=0.001, classifier__neurons=32; total time=   6.8s
[CV] END classifier__activation=relu, classifier__batch_size=16, classifier__dropout_rate=0.0, classifier__epochs=20, classifier__kernel_initializer=uniform, classifier__learning_rate=0.001, classifier__neurons=32; total time=   4.6s
[CV] END classifier__activation=relu, classifier__batch_size=16, classifier__dropout_rate=0.0, classifier__epochs=20, classifier__kernel_initializer=uniform, classifier__learning_rate=0.001, classifier__neurons=32; total time=   2.9s
[CV] END classifier__activation=relu, classifier__batch_size=16,

In [ ]:
print("Best Hyperparameters:")
for parameter, value in grid.best_params_.items():
    print(f"{parameter}: {value}")

print("\nBest Cross-Validation Accuracy:")
print(round(grid.best_score_ * 100, 2), "%")

In [ ]:
results = pd.DataFrame(grid.cv_results_)

top_results = results[
    [
        "rank_test_score",
        "mean_test_score",
        "std_test_score",
        "params"
    ]
].sort_values("rank_test_score").head(5)

display(top_results)

In [ ]:
best_model = grid.best_estimator_

y_pred = best_model.predict(X_test)

print(
    "Test Accuracy:",
    round(accuracy_score(y_test, y_pred) * 100, 2),
    "%"
)

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=["No Diabetes", "Diabetes"],
    zero_division=0
))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
    display_labels=["No Diabetes", "Diabetes"],
    cmap="Blues"
)

plt.title("Confusion Matrix - Best Neural Network")
plt.show()

In [ ]:
print("CONCLUSION")
print("-" * 40)
print("The deep learning model was trained using Keras.")
print("GridSearchCV was used for hyperparameter tuning.")
print("The best parameters were:")

for parameter, value in grid.best_params_.items():
    print(f"{parameter}: {value}")

print(
    "\nBest CV Accuracy:",
    round(grid.best_score_ * 100, 2),
    "%"
)

print(
    "Test Accuracy:",
    round(accuracy_score(y_test, y_pred) * 100, 2),
    "%"
)